# IEEE-CIS Fraud Detection: XGBoost + 特征解释实验

这个 notebook 复用原 LightGBM notebook 的数据和特征工程思路，但把流程整理成更适合学习的版本：

1. 本地读取 IEEE-CIS 数据，并支持小样本 smoke test。
2. 用同一套特征训练 XGBoost。
3. 输出 ROC-AUC、PR-AUC、提交文件和重要性。
4. 用 SHAP、Permutation Importance、消融实验、特征分组、特征精简和协同关系提取理解模型。

默认不跑全量。先确认流程能跑通，再把 `FULL_RUN` 改成 `True` 或逐步增大 `NROWS`。

In [2]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "1_fraud_project":
    candidate = Path("XGBoost/1_fraud_project")
    PROJECT_DIR = candidate if candidate.exists() else PROJECT_DIR
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from fraud_utils.config import OUTPUT_DIR, RANDOM_STATE
from fraud_utils.data import load_ieee_cis_data
from fraud_utils.features import prepare_features
from fraud_utils.modeling import (
    build_xgboost_classifier,
    model_feature_importance,
    run_stratified_cv,
    save_submission,
    summarize_cv_result,
)
from fraud_utils.analysis import (
    average_fold_importance,
    compute_permutation_importance,
    compute_shap_values,
    dependence_candidates,
    sample_for_explanation,
    summarize_importance_by_group,
    two_way_binned_summary,
)
from fraud_utils.experiments import (
    build_feature_group_map,
    compare_feature_sets,
    compute_permutation_synergy,
    joint_permutation_scores,
    permutation_scores_for_features,
    select_top_features,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [3]:
# 小样本优先：先跑通，再逐步增大。
FULL_RUN = False                                    # 是否全量跑
NROWS = None if FULL_RUN else 50_000                # 如果不全量跑数据，阅读的数据集行数
N_SPLITS = 5 if FULL_RUN else 4                     # K 折交叉验证的折数
N_ESTIMATORS = 500 if FULL_RUN else 300             # XGBoost 里树的数量
SHAP_SAMPLE_SIZE = 2_000 if FULL_RUN else 1_000     # SHAP 计算的时候从验证集里抽多少
PERMUTATION_TOP_K = 40 if FULL_RUN else 32          # Permutation Importance 的 TOP-K 

print({
    "FULL_RUN": FULL_RUN,
    "NROWS": NROWS,
    "N_SPLITS": N_SPLITS,
    "N_ESTIMATORS": N_ESTIMATORS,
    "SHAP_SAMPLE_SIZE": SHAP_SAMPLE_SIZE,
})

{'FULL_RUN': False, 'NROWS': 50000, 'N_SPLITS': 4, 'N_ESTIMATORS': 300, 'SHAP_SAMPLE_SIZE': 1000}


## 1. 数据读取与特征工程

这里使用本地 `ieee-fraud-detection` 目录。特征工程保留原 notebook 的主要 Kaggle 思路：UID、金额聚合、邮箱归一化、时间特征、设备/浏览器、频次编码、低信息列删除和 label encoding。

注意：原比赛方案会用 train+test 一起做聚合和频次编码，这对 Kaggle 提交通常有效，但如果迁移到严格业务验证，需要改成只用训练时间窗内已知数据。

In [ ]:
raw = load_ieee_cis_data(nrows=NROWS, optimize_memory=True)
print("raw train:", raw.train.shape)
print("raw test:", raw.test.shape)
print("positive rate:", raw.train["isFraud"].mean())

prepared = prepare_features(raw.train, raw.test, drop_useless=True)
X = prepared.x_train
X_test = prepared.x_test
y = prepared.y

print("X:", X.shape)
print("X_test:", X_test.shape)
print("dropped columns:", len(prepared.metadata["dropped_columns"]))
X.head()

## 2. XGBoost 训练与同口径评估

使用 `StratifiedKFold`，因为 fraud 正样本比例很低。除了 ROC-AUC，也看 PR-AUC，它更能反映不均衡任务下正样本识别能力。

In [ ]:
pos = y.sum()
neg = len(y) - pos
scale_pos_weight = neg / pos

xgb_result = run_stratified_cv(
    model_factory=lambda: build_xgboost_classifier(
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_STATE,
        scale_pos_weight=scale_pos_weight,
    ),
    x=X,
    y=y,
    x_test=X_test,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    model_name="xgboost",
)

summary = summarize_cv_result(xgb_result)
summary

In [ ]:
submission_path = OUTPUT_DIR / ("xgboost_submission_full.csv" if FULL_RUN else "xgboost_submission_smoke.csv")
save_submission(raw.sample_submission.iloc[: len(xgb_result.test_predictions)], xgb_result.test_predictions, submission_path)
print(f"Saved submission -> {submission_path}")

## 3. 重要性：单列与特征组

单列 gain importance 能快速告诉我们模型在哪里分裂最多，但会偏向高基数或容易切分的特征。因此这里再按业务含义聚合到特征组，看 `card`、`C_count`、`D_time_delta`、`uid_aggregate` 等大类的整体贡献。

In [ ]:
importance = average_fold_importance(xgb_result.feature_importance)
importance_path = OUTPUT_DIR / ("xgboost_feature_importance_full.csv" if FULL_RUN else "xgboost_feature_importance_smoke.csv")
importance.to_csv(importance_path, index=False)
print(f"Saved importance -> {importance_path}")

plt.figure(figsize=(10, 12))
sns.barplot(data=importance.head(40), x="importance", y="feature")
plt.title("XGBoost top feature importance")
plt.show()

group_importance = summarize_importance_by_group(importance)
group_importance

## 4. Permutation Importance 与特征协同

Permutation Importance 的问题是：打乱某列后验证分数下降多少。进一步做联合置换时，可以粗略观察两个特征是否互补或冗余：

`synergy = joint_drop - drop_a - drop_b`

- `synergy > 0`：两列一起被破坏时损失更大，可能存在互补关系。
- `synergy < 0`：两列信息可能重叠，联合破坏没有单独损失之和那么大。

这一步默认只在 top 特征上跑，避免 400+ 特征两两组合过慢。

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

candidate_features = importance.head(PERMUTATION_TOP_K)["feature"].tolist()
explain_model = build_xgboost_classifier(
    n_estimators=N_ESTIMATORS,
    random_state=RANDOM_STATE,
    scale_pos_weight=scale_pos_weight,
)
explain_model.fit(X_train[candidate_features], y_train)
valid_pred = explain_model.predict_proba(X_valid[candidate_features])[:, 1]
print("holdout AUC:", roc_auc_score(y_valid, valid_pred))
print("holdout PR-AUC:", average_precision_score(y_valid, valid_pred))

permutation_df = compute_permutation_importance(
    explain_model,
    X_valid[candidate_features],
    y_valid,
    n_repeats=2 if not FULL_RUN else 5,
    random_state=RANDOM_STATE,
)
permutation_df.head(20)

In [ ]:
top_perm_features = permutation_df.head(min(8, len(permutation_df)))["feature"].tolist()
baseline_score, single_scores = permutation_scores_for_features(
    explain_model,
    X_valid[candidate_features],
    y_valid,
    top_perm_features,
    random_state=RANDOM_STATE,
)
joint_scores = joint_permutation_scores(
    explain_model,
    X_valid[candidate_features],
    y_valid,
    top_perm_features,
    max_pairs=20,
    random_state=RANDOM_STATE,
)
synergy_df = compute_permutation_synergy(
    baseline_score=baseline_score,
    single_scores=single_scores,
    joint_scores=joint_scores,
)
synergy_df.head(20)

## 5. SHAP：全局方向、单条样本和依赖图

SHAP 解释的是“当前模型如何使用特征”，不是因果关系。对大数据全量计算会很慢，所以默认抽样。建议先看 summary/bar，再挑 3-5 个业务上重要的特征做 dependence plot。

In [ ]:
x_shap = sample_for_explanation(
    X_valid[candidate_features],
    sample_size=SHAP_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
)
explainer, shap_values = compute_shap_values(explain_model, x_shap)

import shap
shap.summary_plot(shap_values, x_shap, plot_type="bar", show=False)
plt.title("SHAP global importance")
plt.show()

shap.summary_plot(shap_values, x_shap, show=False)
plt.title("SHAP summary")
plt.show()

In [ ]:
for feature in dependence_candidates(importance, top_k=5):
    if feature not in x_shap.columns:
        continue
    shap.dependence_plot(feature, shap_values, x_shap, show=False)
    plt.title(f"SHAP dependence: {feature}")
    plt.show()

## 6. 消融实验与特征精简

消融实验回答：去掉某一组特征后，指标掉多少？

特征精简回答：能否用更少特征接近 full model 的效果？这对理解模型、控制训练速度和降低线上特征成本都很重要。

In [ ]:
feature_groups = build_feature_group_map(candidate_features)
feature_sets = {"candidate_top_features": candidate_features}

# 对 top 特征做轻量组消融：每次移除一个特征组，重训候选模型。
for group, columns in feature_groups.items():
    kept = [feature for feature in candidate_features if feature not in set(columns)]
    if len(kept) >= 2:
        feature_sets[f"drop_group_{group}"] = kept

ablation_df = compare_feature_sets(
    model_factory=lambda: build_xgboost_classifier(
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_STATE,
        scale_pos_weight=scale_pos_weight,
    ),
    x_train=X_train,
    y_train=y_train,
    x_valid=X_valid,
    y_valid=y_valid,
    feature_sets=feature_sets,
)
ablation_df

In [ ]:
top_50 = select_top_features(importance, top_k=50, always_keep=["TransactionAmt"])
top_100 = select_top_features(importance, top_k=100, always_keep=["TransactionAmt"])
positive_perm = permutation_df.loc[permutation_df["importance"] > 0, "feature"].tolist()

pruning_sets = {
    "top_50": [feature for feature in top_50 if feature in X.columns],
    "top_100": [feature for feature in top_100 if feature in X.columns],
    "positive_permutation": [feature for feature in positive_perm if feature in X.columns],
}
pruning_sets = {name: features for name, features in pruning_sets.items() if len(features) >= 2}

pruning_df = compare_feature_sets(
    model_factory=lambda: build_xgboost_classifier(
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_STATE,
        scale_pos_weight=scale_pos_weight,
    ),
    x_train=X_train,
    y_train=y_train,
    x_valid=X_valid,
    y_valid=y_valid,
    feature_sets=pruning_sets,
)
pruning_df

## 7. 协同关系候选提取

把联合置换分数、SHAP dependence 和二维分桶统计结合起来看。下面先输出候选关系表，再示例画一个二维分桶的 fraud rate。

In [ ]:
relation_candidates = synergy_df.merge(
    permutation_df[["feature", "importance"]].rename(columns={"feature": "feature_a", "importance": "perm_importance_a"}),
    on="feature_a",
    how="left",
).merge(
    permutation_df[["feature", "importance"]].rename(columns={"feature": "feature_b", "importance": "perm_importance_b"}),
    on="feature_b",
    how="left",
)
relation_path = OUTPUT_DIR / ("xgboost_relation_candidates_full.csv" if FULL_RUN else "xgboost_relation_candidates_smoke.csv")
relation_candidates.to_csv(relation_path, index=False)
print(f"Saved relation candidates -> {relation_path}")
relation_candidates.head(20)

In [ ]:
if not relation_candidates.empty:
    feature_a = relation_candidates.loc[0, "feature_a"]
    feature_b = relation_candidates.loc[0, "feature_b"]
    analysis_frame = X[[feature_a, feature_b]].copy()
    analysis_frame["isFraud"] = y
    binned = two_way_binned_summary(analysis_frame, feature_a, feature_b, "isFraud", bins=8)
    display(binned.head(20))